# Lendo o arquivo bruto

In [0]:
# Lendo o arquivo csv que está no datalake
produtos = spark.read.format("csv").option("header","true").option("delimiter",",").load("/Volumes/lh_nautical/datalake/datas/produtos_raw.csv")
#Contando as linhas do arquivo
produtosLines = produtos.count()
#Mostrando todo o conteúdo do arquivo
produtos.show(produtosLines)




# Criando os Schemas  


In [0]:
%sql
-- Informando o catalogo
USE CATALOG lh_nautical;
SHOW VOLUMES;
-- Criando o schema Bronze caso ele não exista
CREATE SCHEMA IF NOT EXISTS bronze_lh_nautical;
CREATE SCHEMA IF NOT EXISTS silver_lh_nautical;
CREATE SCHEMA IF NOT EXISTS gold_lh_nautical;


# Salvando o arquivo bruto como tabela delta na camada Bronze

In [0]:
# Salvando o arquivo como tabela na camada bronze 
produtos.write.mode("overwrite").format("delta").saveAsTable("bronze_lh_nautical.tbl_produtos")

# Conferindo a tabela delta criada na camada Bronze


In [0]:
#Conferindo o conteúdo da tabela
tbl_produtos = spark.read.table("bronze_lh_nautical.tbl_produtos")
tbl_produtos.show()

# Recriando a tabela para camada Silver



### - Separando a unidade monetaria do preço


In [0]:
# importacão de algumas bibliotecas
from pyspark.sql.functions import substring, col
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Lendo a tabela que está na camada bronze
tbl_produtos_clean = spark.read.table("bronze_lh_nautical.tbl_produtos")
# Informando qual o divisor para separar o preço e a moeda
split = F.split(tbl_produtos_clean['price'], " ")
#Separando o preço e a moeda
tbl_produtos_clean  = tbl_produtos_clean.withColumn('monetary_unit', split.getItem(0)).withColumn('price', split.getItem(1))
#Conferindo o conteúdo da tabela
tbl_produtos_clean.show()



### - Modificando o tipo das colunas price e code

In [0]:
#Importando algumas bibliotecas
from pyspark.sql.functions import when, col
from pyspark.sql.types import *

#Verificando o tipo das colunas
tbl_produtos_clean.printSchema()
#Modificando os tipos das colunas para um tipo mais adequado para cada informação e salvando a tabela na camada silver
tbl_produtos_clean.withColumn('code', col("code").cast(IntegerType())).withColumn('price', col("price").cast(DecimalType(10,2))).write.mode("overwrite").format("delta").saveAsTable("silver_lh_nautical.tbl_produtos")
#Conferindo o conteúdo da tabela
tbl_produtos_clean = spark.read.table("silver_lh_nautical.tbl_produtos")
tbl_produtos_clean.show()


### - Tratando os dados da coluna actual_category

In [0]:
from pyspark.sql.functions import col, trim, lower, regexp_replace
from pyspark.sql.functions import when, col
#Tratando os dados
tbl_produtos_clean =tbl_produtos_clean.withColumn("actual_category", regexp_replace(lower(trim(col("actual_category"))), "[^a-zA-Z0-9]", ""))

#Padronizando os dados da coluna actual_category
tbl_produtos_clean.select("actual_category").show(produtosLines)
tbl_produtos_clean = tbl_produtos_clean.withColumn("actual_category", when(tbl_produtos_clean.actual_category.startswith("ele"), "Eletrônicos").when(tbl_produtos_clean.actual_category.startswith("prop"), "Propulsão").when(tbl_produtos_clean.actual_category.startswith("anc") | tbl_produtos_clean.actual_category.startswith("en"), "Ancoragem"))

#conferindo
tbl_produtos_clean.select("actual_category").distinct().show()

#Salvando a modificação
tbl_produtos_clean.write.mode("overwrite").format("delta").saveAsTable("silver_lh_nautical.tbl_produtos")
tbl_produtos_clean = spark.read.table("silver_lh_nautical.tbl_produtos")
tbl_produtos_clean.show(produtosLines)


